# Baseline: Hough transform + MLP/CNN

Two-stage classical baseline. A circular Hough transform locates the sign, and a small neural network classifies the cropped region into one of the six classes. The classifiers are trained on a synthetic dataset generated from one reference image per class.

1. Setup
2. Synthetic dataset
3. Training
4. Detection pipeline
5. Evaluation and export

## 1. Setup

In [ ]:
import time
import urllib.request

import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
rng = np.random.default_rng(SEED)

SIZE = 80                                 # network input, 80x80 grayscale
MARGIN = 0.25                             # crop margin around the detected circle, as a fraction of the radius
MASK_RADIUS = SIZE / (2 * (1 + MARGIN))   # sign radius inside the 80x80 crop (32 px)

CLASSES = [
    "R-25a_esquerda",
    "R-25b_direita",
    "R-26_frente",
    "R-3_sentido-proibido",
    "R-4a_proibido-esquerda",
    "R-4b_proibido-direita",
]
N_CLASSES = len(CLASSES)

print("OpenCV    :", cv2.__version__)
print("TensorFlow:", tf.__version__)
print("GPU       :", tf.config.list_physical_devices("GPU") or "none")

## 2. Synthetic dataset

The reference images `PL1.png` to `PL6.png` are downloaded from the repository. Each one is converted to grayscale, resized to 80x80 and augmented 400 times, giving 2400 images.

| Transformation | Range |
|---|---|
| Rotation | ±5° |
| Scale | ±20% |
| Shift | ±3 px |
| Brightness | ±30 gray levels |
| Gaussian blur | kernel 1, 3 or 5 |
| Gaussian noise | σ = 8 |

There is no horizontal flip, since it would turn left signs into right signs. The border fill is black, matching the background of the reference images.

In [ ]:
BASE_URL = ("https://raw.githubusercontent.com/vitor-domeniquelli/"
            "brazilian-traffic-sign-detection/main/data/reference_signs")

# File number for each class, in the order of CLASSES
FILE_ORDER = [2, 3, 1, 4, 6, 5]

references_bgr = []
for n in FILE_ORDER:
    path, _ = urllib.request.urlretrieve(f"{BASE_URL}/PL{n}.png", f"PL{n}.png")
    references_bgr.append(cv2.imread(path))

# Check that each image matches its class name
fig, axes = plt.subplots(1, N_CLASSES, figsize=(15, 3))
for ax, img, name in zip(axes, references_bgr, CLASSES):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(name, fontsize=9)
    ax.axis("off")
plt.show()

In [ ]:
SAMPLES_PER_CLASS = 400

ROT_MAX = 5.0
SCALE_MAX = 0.20
SHIFT_MAX = 3.0
BRIGHTNESS_MAX = 30.0
BLURS = [1, 3, 5]
NOISE_SIGMA = 8.0
BACKGROUND = 0

REFERENCES = [cv2.resize(cv2.cvtColor(im, cv2.COLOR_BGR2GRAY), (SIZE, SIZE),
                         interpolation=cv2.INTER_AREA) for im in references_bgr]


def geometric(img, angle=0.0, scale=1.0, dx=0.0, dy=0.0):
    # Rotation, scale and shift in a single affine transform
    M = cv2.getRotationMatrix2D((SIZE / 2, SIZE / 2), angle, scale)
    M[0, 2] += dx
    M[1, 2] += dy
    return cv2.warpAffine(img, M, (SIZE, SIZE), flags=cv2.INTER_LINEAR, borderValue=BACKGROUND)


def photometric(img, brightness=0.0, blur=1, sigma=0.0):
    if brightness:
        img = cv2.convertScaleAbs(img, alpha=1.0, beta=brightness)
    if blur > 1:
        img = cv2.GaussianBlur(img, (blur, blur), 0)
    if sigma > 0:
        img = np.clip(img.astype(np.float32) + rng.normal(0, sigma, img.shape), 0, 255).astype(np.uint8)
    return img


def augment(base):
    img = geometric(base,
                    angle=rng.uniform(-ROT_MAX, ROT_MAX),
                    scale=1.0 + rng.uniform(-SCALE_MAX, SCALE_MAX),
                    dx=rng.uniform(-SHIFT_MAX, SHIFT_MAX),
                    dy=rng.uniform(-SHIFT_MAX, SHIFT_MAX))
    return photometric(img,
                       brightness=rng.uniform(-BRIGHTNESS_MAX, BRIGHTNESS_MAX),
                       blur=int(rng.choice(BLURS)),
                       sigma=NOISE_SIGMA)


N_TOTAL = SAMPLES_PER_CLASS * N_CLASSES
X = np.empty((N_TOTAL, SIZE, SIZE), dtype=np.uint8)
y = np.empty(N_TOTAL, dtype=np.int64)

k = 0
for label, base in enumerate(REFERENCES):
    for _ in range(SAMPLES_PER_CLASS):
        X[k] = augment(base)
        y[k] = label
        k += 1

print("dataset:", X.shape, "| per class:", np.bincount(y))

In [ ]:
# Stratified 80/10/10 split
X_train, X_rest, y_train, y_rest = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(
    X_rest, y_rest, test_size=0.50, stratify=y_rest, random_state=SEED)


def normalize(a):
    return (a.astype(np.float32) / 255.0).reshape(-1, SIZE, SIZE, 1)


X_train_n, X_val_n, X_test_n = normalize(X_train), normalize(X_val), normalize(X_test)
print(f"train {len(y_train)} | val {len(y_val)} | test {len(y_test)}")

## 3. Training

Both networks use the same optimizer, callbacks and data.

| Model | Architecture |
|---|---|
| MLP | 5 dense layers of 512 units |
| CNN | 2 × [Conv(32, 3×3) → MaxPool(2×2)] → Dense(512) |

In [ ]:
EPOCHS = 40
BATCH = 32
LEARNING_RATE = 1e-3


def build_mlp():
    return tf.keras.Sequential([
        tf.keras.layers.Input((SIZE, SIZE, 1)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(512, activation="relu"),
        tf.keras.layers.Dense(512, activation="relu"),
        tf.keras.layers.Dense(512, activation="relu"),
        tf.keras.layers.Dense(512, activation="relu"),
        tf.keras.layers.Dense(512, activation="relu"),
        tf.keras.layers.Dense(N_CLASSES, activation="softmax"),
    ])


def build_cnn():
    return tf.keras.Sequential([
        tf.keras.layers.Input((SIZE, SIZE, 1)),
        tf.keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(512, activation="relu"),
        tf.keras.layers.Dense(N_CLASSES, activation="softmax"),
    ])


def train(build):
    model = build()
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
                  loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                  metrics=["accuracy"])
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8,
                                         min_delta=1e-3, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2,
                                             patience=3, min_delta=1e-3),
    ]
    model.fit(X_train_n, y_train, validation_data=(X_val_n, y_val),
              epochs=EPOCHS, batch_size=BATCH, callbacks=callbacks, verbose=2)
    return model

In [ ]:
mlp = train(build_mlp)

In [ ]:
cnn = train(build_cnn)

## 4. Detection pipeline

grayscale → median filter → circular Hough transform → square crop → resize to 80x80 → Otsu threshold → circular mask → classifier

In [ ]:
MIN_DIST = 60
PARAM1 = 120
PARAM2_STEPS = [0.98, 0.95, 0.90, 0.85]   # relaxed in steps when no circle is found
RADIUS_MIN, RADIUS_MAX = 10, 80

_yy, _xx = np.ogrid[:SIZE, :SIZE]
_c = (SIZE - 1) / 2.0
OUTSIDE_MASK = ((_xx - _c) ** 2 + (_yy - _c) ** 2) > MASK_RADIUS ** 2


def find_circle(gray):
    # Returns the largest circle (cx, cy, r), or None
    for p2 in PARAM2_STEPS:
        circles = cv2.HoughCircles(gray, cv2.HOUGH_GRADIENT_ALT, 1, MIN_DIST,
                                   param1=PARAM1, param2=p2,
                                   minRadius=RADIUS_MIN, maxRadius=RADIUS_MAX)
        if circles is not None:
            break
    if circles is None:
        return None
    # Plain int avoids unsigned overflow in cx - r near the image border
    circles = [tuple(int(v) for v in c) for c in np.round(circles[0]).astype(int)]
    return max(circles, key=lambda c: c[2])


def crop_sign(gray, cx, cy, r):
    # Square window around the circle, padded by replication where it leaves the image
    half = int(round(r * (1 + MARGIN)))
    x0, y0, x1, y1 = cx - half, cy - half, cx + half, cy + half
    pad_left, pad_top = max(0, -x0), max(0, -y0)
    pad_right = max(0, x1 - gray.shape[1])
    pad_bottom = max(0, y1 - gray.shape[0])

    crop = gray[max(y0, 0):min(y1, gray.shape[0]), max(x0, 0):min(x1, gray.shape[1])]
    if crop.size == 0:
        return None, (x0, y0, x1, y1)
    if pad_left or pad_top or pad_right or pad_bottom:
        crop = cv2.copyMakeBorder(crop, pad_top, pad_bottom, pad_left, pad_right,
                                  cv2.BORDER_REPLICATE)
    return crop, (x0, y0, x1, y1)


def preprocess(crop):
    img = cv2.resize(crop, (SIZE, SIZE), interpolation=cv2.INTER_AREA)
    _, img = cv2.threshold(img, 0, 255, cv2.THRESH_OTSU)
    img[OUTSIDE_MASK] = 0
    return img


def detect_and_classify(image_bgr, model):
    # Returns (class, confidence, box, timings in s), or None if no circle is found
    t0 = time.perf_counter()
    gray = cv2.medianBlur(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY), 3)
    circle = find_circle(gray)
    t_hough = time.perf_counter() - t0
    if circle is None:
        return None

    crop, box = crop_sign(gray, *circle)
    if crop is None:
        return None
    t0 = time.perf_counter()
    x = (preprocess(crop).astype(np.float32) / 255.0).reshape(1, SIZE, SIZE, 1)
    t_prep = time.perf_counter() - t0

    t0 = time.perf_counter()
    probs = np.asarray(model(x, training=False))[0]
    t_infer = time.perf_counter() - t0

    timings = {"hough": t_hough, "preprocess": t_prep, "inference": t_infer}
    return int(np.argmax(probs)), float(np.max(probs)), box, timings

## 5. Evaluation and export

In [ ]:
# Classifier accuracy on the synthetic test split
for name, model in [("MLP", mlp), ("CNN", cnn)]:
    y_pred = np.argmax(model.predict(X_test_n, verbose=0), axis=1)
    print(f"===== {name} ({model.count_params():,} parameters)")
    print(classification_report(y_test, y_pred, target_names=CLASSES, digits=4, zero_division=0))

In [ ]:
mlp.save("mlp.keras")
cnn.save("cnn.keras")

from google.colab import files
files.download("mlp.keras")
files.download("cnn.keras")